In [1]:
import random

demands = [12,45,23,67,34,19,56,38,72,15,49,61,27,83,41,55,30,77,
64,18,52,39,71,26,44,91,33,58,22,85,16,69,47,74,31,53]

def calc_value(state, d):
    return sum(d[i] for i in state) - 5 * len(state)

def generate_neighbors(state):
    neighbors = []
    remaining = set(range(36)) - state
    for s in state:
        for r in remaining:
            new_state = set(state)
            new_state.remove(s)
            new_state.add(r)
            neighbors.append(new_state)
    return neighbors

def create_state():
    return set(random.sample(range(36), 10))


for _ in range(3):
    st = create_state()
    neigh = generate_neighbors(st)
    print("State:", st)
    print("Fitness:", calc_value(st, demands))
    print("Neighbours:", len(neigh))
    print("Valid:", all(len(n)==10 and len(n)==len(set(n)) for n in neigh))
    print()

State: {33, 4, 10, 11, 14, 16, 22, 25, 27, 28}
Fitness: 481
Neighbours: 260
Valid: True

State: {32, 33, 3, 8, 9, 13, 15, 21, 25, 27}
Fitness: 551
Neighbours: 260
Valid: True

State: {0, 34, 4, 5, 9, 12, 13, 17, 24, 30}
Fitness: 308
Neighbours: 260
Valid: True



In [2]:
#part b
def hc_run(state, d, mode):
    current = state
    steps = 0

    while True:
        neighbors = generate_neighbors(current)

        if mode == "first":
            next_state = None
            for n in neighbors:
                if calc_value(n, d) > calc_value(current, d):
                    next_state = n
                    break
        else:
            better = [n for n in neighbors if calc_value(n, d) > calc_value(current, d)]
            next_state = random.choice(better) if better else None

        if not next_state:
            return current, calc_value(current, d), steps

        current = next_state
        steps += 1


def rrhc_run(restarts, d, mode):
    best_state = None
    best_val = float('-inf')
    results = []

    for _ in range(restarts):
        st = create_state()
        final, val, _ = hc_run(st, d, mode)
        results.append(val)

        if val > best_val:
            best_val = val
            best_state = final

    return best_state, best_val, results


best_s, best_v, vals = rrhc_run(30, demands, "stochastic")

print("Best State:", best_s)
print("Fitness:", best_v)
print("Zones:", [(i//6, i%6) for i in best_s])
print("Per restart:", vals)

Best State: {33, 3, 8, 13, 17, 18, 22, 25, 29, 31}
Fitness: 703
Zones: [(5, 3), (0, 3), (1, 2), (2, 1), (2, 5), (3, 0), (3, 4), (4, 1), (4, 5), (5, 1)]
Per restart: [703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703]


In [3]:
#part c
def ga_value(chrom, d):
    return sum(d[i] for i in chrom) - 5 * len(chrom)

def ox(p1, p2):
    a, b = sorted(random.sample(range(10), 2))
    child = [-1]*10
    child[a:b] = p1[a:b]

    fill = [x for x in p2 if x not in child]
    j = 0
    for i in range(10):
        if child[i] == -1:
            child[i] = fill[j]
            j += 1
    return child

def ga_mutate(chrom, pm):
    if random.random() < pm:
        new = chrom[:]
        idx = random.randint(0,9)
        available = list(set(range(36)) - set(new))
        new[idx] = random.choice(available)
        return sorted(new)
    return chrom

def tournament(pop):
    cand = random.sample(pop, 3)
    return max(cand, key=lambda x: ga_value(x, demands))

def run_ga(size, gens, pm):
    pop = [sorted(random.sample(range(36),10)) for _ in range(size)]

    for _ in range(gens):
        new_pop = []
        while len(new_pop) < size:
            p1 = tournament(pop)
            p2 = tournament(pop)

            c = ox(p1, p2)
            c = ga_mutate(c, pm)

            new_pop.append(c)

        pop = new_pop

    best = max(pop, key=lambda x: ga_value(x, demands))
    return best, ga_value(best, demands)


best_ga, val_ga = run_ga(30, 100, 0.1)

print("Best GA:", best_ga)
print("Fitness:", val_ga)
print("Zones:", [(i//6, i%6) for i in best_ga])

Best GA: [8, 22, 25, 13, 3, 29, 6, 11, 17, 31]
Fitness: 682
Zones: [(1, 2), (3, 4), (4, 1), (2, 1), (0, 3), (4, 5), (1, 0), (1, 5), (2, 5), (5, 1)]


In [4]:
#part d
import statistics

rrhc_results = []
ga_results = []

for _ in range(20):
    _, v1, _ = rrhc_run(30, demands, "stochastic")
    _, v2 = run_ga(30, 100, 0.1)

    rrhc_results.append(v1)
    ga_results.append(v2)

print("\nRRHC Mean:", statistics.mean(rrhc_results))
print("RRHC Std:", statistics.stdev(rrhc_results))
print("RRHC Best:", max(rrhc_results))

print("\nGA Mean:", statistics.mean(ga_results))
print("GA Std:", statistics.stdev(ga_results))
print("GA Best:", max(ga_results))


RRHC Mean: 703
RRHC Std: 0.0
RRHC Best: 703

GA Mean: 694.6
GA Std: 6.021015826166493
GA Best: 703
